# NB12: ModelSEED Template Reaction Coverage

**Purpose**: Parse the ModelSEED reconstruction templates (v7.0) and determine what
fraction of template reactions are covered by the Rosetta multi-evidence pipeline.
Template reactions are classified as conditional, gapfilling, spontaneous, or universal —
stratify coverage by type. Also characterize the 15K+ UniProt-mapped reactions that
fall outside any template and assess detection strategies.

**Context**: ModelSEED templates define the reaction universe for automated metabolic
model reconstruction. Conditional reactions are added when the corresponding enzyme is
detected in the genome (via RAST). Gapfilling reactions are added computationally to
ensure model feasibility. Spontaneous reactions proceed without enzymes. Universal
reactions are always included.

**Requires**: Local data only (no Spark)

**Output**: `template_coverage.png`, `template_coverage_summary.tsv`

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

DATA_DIR = '../data'
FIG_DIR = '../figures'
USER_DIR = '../user_data'
TEMPLATE_DIR = f'{USER_DIR}/ModelSEEDTemplates/templates/v7.0'

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
})

## 1. Parse Template Reactions

Load all four v7.0 templates and extract reaction IDs, types, and complex/role associations.

In [2]:
template_files = {
    'GramNeg': 'GramNegModelTemplateV7.json',
    'GramPos': 'GramPosModelTemplateV7.json',
    'Core': 'Core-V6.json',
    'Archaea': 'ArchaeaTemplateV6.json',
}

rxn_type_map = {}
rxn_templates = defaultdict(set)
per_template_types = defaultdict(lambda: defaultdict(set))
template_metadata = {}

for label, fname in template_files.items():
    with open(f'{TEMPLATE_DIR}/{fname}') as f:
        d = json.load(f)
    template_metadata[label] = {
        'name': d.get('name', ''),
        'domain': d.get('domain', ''),
        'n_reactions': len(d['reactions']),
        'n_complexes': len(d.get('complexes', [])),
        'n_roles': len(d.get('roles', [])),
    }
    for r in d['reactions']:
        bare = r['id'].rsplit('_', 1)[0]
        rtype = r['type']
        rxn_type_map[bare] = rtype
        rxn_templates[bare].add(label)
        per_template_types[label][rtype].add(bare)

all_template_rxns = set(rxn_type_map.keys())

print('Template inventory:')
for label, meta in template_metadata.items():
    types_str = ', '.join(f'{t}={len(per_template_types[label][t])}'
                          for t in ['conditional', 'gapfilling', 'spontaneous', 'universal']
                          if per_template_types[label][t])
    print(f'  {label}: {meta["n_reactions"]} reactions, '
          f'{meta["n_complexes"]} complexes, {meta["n_roles"]} roles')
    print(f'    domain={meta["domain"]}, types: {types_str}')

print(f'\nUnion across all templates: {len(all_template_rxns)} unique reactions')

type_sets = defaultdict(set)
for rxn, rtype in rxn_type_map.items():
    type_sets[rtype].add(rxn)

print(f'\nType distribution (union):')
for t in ['conditional', 'gapfilling', 'spontaneous', 'universal']:
    print(f'  {t}: {len(type_sets[t])}')

print(f'\nTemplate overlap:')
labels = sorted(template_files.keys())
for i, t1 in enumerate(labels):
    rxns1 = set()
    for rtype_set in per_template_types[t1].values():
        rxns1 |= rtype_set
    for t2 in labels[i+1:]:
        rxns2 = set()
        for rtype_set in per_template_types[t2].values():
            rxns2 |= rtype_set
        shared = rxns1 & rxns2
        only1 = rxns1 - rxns2
        only2 = rxns2 - rxns1
        print(f'  {t1} vs {t2}: {len(shared)} shared, '
              f'{len(only1)} {t1}-only, {len(only2)} {t2}-only')

Template inventory:
  GramNeg: 8584 reactions, 3296 complexes, 20548 roles
    domain=Bacteria, types: conditional=6284, gapfilling=2258, spontaneous=31, universal=11
  GramPos: 8584 reactions, 3296 complexes, 20548 roles
    domain=Bacteria, types: conditional=6284, gapfilling=2258, spontaneous=31, universal=11
  Core: 252 reactions, 468 complexes, 706 roles
    domain=Bacteria, types: conditional=230, gapfilling=4, spontaneous=13, universal=5
  Archaea: 8600 reactions, 3322 complexes, 20553 roles
    domain=Bacteria, types: conditional=6303, gapfilling=2255, spontaneous=31, universal=11

Union across all templates: 8606 unique reactions

Type distribution (union):
  conditional: 6306
  gapfilling: 2258
  spontaneous: 31
  universal: 11

Template overlap:
  Archaea vs Core: 249 shared, 8351 Archaea-only, 3 Core-only
  Archaea vs GramNeg: 8578 shared, 22 Archaea-only, 6 GramNeg-only
  Archaea vs GramPos: 8578 shared, 22 Archaea-only, 6 GramPos-only
  Core vs GramNeg: 252 shared, 0 Core

## 2. Load Evidence Data

Load the evidence integration summary, transport evidence, EC-to-reaction bridge,
and RAST protein-EC pairs from prior notebooks.

In [3]:
evidence = pd.read_parquet(f'{DATA_DIR}/evidence_integration_summary.parquet')
balanced_ids = set(evidence['rxn_bare'])
mapped_ids = set(evidence[evidence['any_evidence']]['rxn_bare'])

transport_ev = pd.read_parquet(f'{DATA_DIR}/transport_evidence_mapping.parquet')
transport_mapped = set(transport_ev['rxn_bare'])
all_mapped = mapped_ids | transport_mapped

ec_to_rxn = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
all_ec_rxns = set(ec_to_rxn['rxn_bare'].unique())

rast_ec = pd.read_parquet(f'{DATA_DIR}/rast_protein_ec.parquet')
rast_ecs = set(rast_ec['ec'].unique())
rast_rxns = set(ec_to_rxn[ec_to_rxn['ec'].isin(rast_ecs)]['rxn_bare'].unique())

transport_analysis = pd.read_parquet(f'{DATA_DIR}/transport_analysis.parquet')
transport_rxn_ids = set(transport_analysis[transport_analysis['is_transport']]['rxn_bare'])

print(f'Balanced ModelSEED reactions: {len(balanced_ids):,}')
print(f'EC-based evidence (Tiers 1-3 + RAST): {len(mapped_ids):,}')
print(f'Transport evidence (NB11): {len(transport_mapped):,}')
print(f'All evidence (union): {len(all_mapped):,}')
print(f'RAST-reachable reactions: {len(rast_rxns):,}')
print(f'Reactions with EC in bridge table: {len(all_ec_rxns):,}')

Balanced ModelSEED reactions: 34,343
EC-based evidence (Tiers 1-3 + RAST): 17,351
Transport evidence (NB11): 4,627
All evidence (union): 21,254
RAST-reachable reactions: 10,730
Reactions with EC in bridge table: 18,660


## 3. Template Coverage by Reaction Type

For each template reaction type (conditional, gapfilling, spontaneous, universal),
compute coverage against the evidence pipeline and RAST.

In [4]:
rows = []
for rtype in ['conditional', 'gapfilling', 'spontaneous', 'universal']:
    rxns = type_sets[rtype]
    bal = rxns & balanced_ids
    ev_ec = rxns & mapped_ids
    ev_transport = rxns & transport_mapped
    ev_any = rxns & all_mapped
    gap = bal - all_mapped
    unbal = rxns - balanced_ids
    rast = rxns & rast_rxns

    rows.append({
        'type': rtype,
        'total': len(rxns),
        'balanced': len(bal),
        'unbalanced': len(unbal),
        'ec_evidence': len(ev_ec),
        'transport_evidence': len(ev_transport),
        'any_evidence': len(ev_any),
        'evidence_pct': 100 * len(ev_any) / len(rxns) if rxns else 0,
        'balanced_evidence_pct': 100 * len(ev_any) / len(bal) if bal else 0,
        'gap': len(gap),
        'gap_pct_of_balanced': 100 * len(gap) / len(bal) if bal else 0,
        'rast_reachable': len(rast),
        'rast_pct': 100 * len(rast) / len(rxns) if rxns else 0,
    })

totals = {
    'type': 'TOTAL',
    'total': len(all_template_rxns),
    'balanced': len(all_template_rxns & balanced_ids),
    'unbalanced': len(all_template_rxns - balanced_ids),
    'ec_evidence': len(all_template_rxns & mapped_ids),
    'transport_evidence': len(all_template_rxns & transport_mapped),
    'any_evidence': len(all_template_rxns & all_mapped),
    'evidence_pct': 100 * len(all_template_rxns & all_mapped) / len(all_template_rxns),
    'balanced_evidence_pct': 100 * len(all_template_rxns & all_mapped) / len(all_template_rxns & balanced_ids),
    'gap': len((all_template_rxns & balanced_ids) - all_mapped),
    'gap_pct_of_balanced': 100 * len((all_template_rxns & balanced_ids) - all_mapped) / len(all_template_rxns & balanced_ids),
    'rast_reachable': len(all_template_rxns & rast_rxns),
    'rast_pct': 100 * len(all_template_rxns & rast_rxns) / len(all_template_rxns),
}
rows.append(totals)

coverage_df = pd.DataFrame(rows)

print('Template Reaction Coverage by Type')
print('=' * 100)
print(f'{"Type":<14} {"Total":>6} {"Balanced":>9} {"EC Ev":>7} {"Trans":>7} {"Any Ev":>7} '
      f'{"Ev%":>7} {"BalEv%":>7} {"RAST":>6} {"Gap":>6} {"Gap%":>6}')
print('-' * 100)
for _, r in coverage_df.iterrows():
    print(f'{r["type"]:<14} {r["total"]:>6} {r["balanced"]:>9} {r["ec_evidence"]:>7} '
          f'{r["transport_evidence"]:>7} {r["any_evidence"]:>7} '
          f'{r["evidence_pct"]:>6.1f}% {r["balanced_evidence_pct"]:>6.1f}% '
          f'{r["rast_reachable"]:>6} {r["gap"]:>6} {r["gap_pct_of_balanced"]:>5.1f}%')

Template Reaction Coverage by Type
Type            Total  Balanced   EC Ev   Trans  Any Ev     Ev%  BalEv%   RAST    Gap   Gap%
----------------------------------------------------------------------------------------------------
conditional      6306      5770    4813     281    5043   80.0%   87.4%   3108    727  12.6%
gapfilling       2258      2059     514     362     858   38.0%   41.7%    380   1201  58.3%
spontaneous        31        29       2       6       8   25.8%   27.6%      1     21  72.4%
universal          11        11       1       7       7   63.6%   63.6%      1      4  36.4%
TOTAL            8606      7869    5330     656    5916   68.7%   75.2%   3490   1953  24.8%


## 4. Gap Analysis: Template Reactions Without Evidence

For balanced template reactions with no evidence, characterize why:
EC orphan (no EC assigned), has EC but no protein, or transport reaction.

In [5]:
print('Gap Analysis: Balanced Template Reactions Without Evidence')
print('=' * 80)

gap_rows = []
for rtype in ['conditional', 'gapfilling', 'spontaneous', 'universal']:
    gap = (type_sets[rtype] & balanced_ids) - all_mapped
    if not gap:
        continue

    gap_has_ec = gap & all_ec_rxns
    gap_no_ec = gap - all_ec_rxns
    gap_transport = gap & transport_rxn_ids
    gap_non_transport = gap - transport_rxn_ids
    gap_rast = gap & rast_rxns

    gap_rows.append({
        'type': rtype,
        'gap_total': len(gap),
        'has_ec_no_protein': len(gap_has_ec),
        'ec_orphan': len(gap_no_ec),
        'transport': len(gap_transport),
        'non_transport': len(gap_non_transport),
        'rast_reachable': len(gap_rast),
    })

    print(f'\n--- {rtype.upper()} ({len(gap)} gap reactions) ---')
    print(f'  Has EC, no protein found: {len(gap_has_ec)}')
    print(f'  No EC assigned (orphan):  {len(gap_no_ec)}')
    print(f'  Transport reactions:      {len(gap_transport)}')
    print(f'  Non-transport:            {len(gap_non_transport)}')
    print(f'  RAST-reachable:           {len(gap_rast)}')

gap_df = pd.DataFrame(gap_rows)
gap_totals = gap_df[['gap_total', 'has_ec_no_protein', 'ec_orphan',
                      'transport', 'non_transport', 'rast_reachable']].sum()
print(f'\n--- TOTAL ({int(gap_totals["gap_total"])} gap reactions) ---')
print(f'  Has EC, no protein found: {int(gap_totals["has_ec_no_protein"])}')
print(f'  No EC assigned (orphan):  {int(gap_totals["ec_orphan"])}')
print(f'  Transport reactions:      {int(gap_totals["transport"])}')
print(f'  Non-transport:            {int(gap_totals["non_transport"])}')
print(f'  RAST-reachable:           {int(gap_totals["rast_reachable"])}')

Gap Analysis: Balanced Template Reactions Without Evidence

--- CONDITIONAL (727 gap reactions) ---
  Has EC, no protein found: 560
  No EC assigned (orphan):  167
  Transport reactions:      27
  Non-transport:            700
  RAST-reachable:           0

--- GAPFILLING (1201 gap reactions) ---
  Has EC, no protein found: 19
  No EC assigned (orphan):  1182
  Transport reactions:      95
  Non-transport:            1106
  RAST-reachable:           0

--- SPONTANEOUS (21 gap reactions) ---
  Has EC, no protein found: 1
  No EC assigned (orphan):  20
  Transport reactions:      9
  Non-transport:            12
  RAST-reachable:           0

--- UNIVERSAL (4 gap reactions) ---
  Has EC, no protein found: 0
  No EC assigned (orphan):  4
  Transport reactions:      4
  Non-transport:            0
  RAST-reachable:           0

--- TOTAL (1953 gap reactions) ---
  Has EC, no protein found: 580
  No EC assigned (orphan):  1373
  Transport reactions:      135
  Non-transport:            1818

## 5. Non-Template Reactions With Evidence

Characterize the UniProt-mapped reactions that are NOT in any reconstruction template —
potential model expansion candidates. Assess how they would be detected in a genome.

In [6]:
mapped_not_template = all_mapped - all_template_rxns
mnt_rast = mapped_not_template & rast_rxns
mnt_no_rast = mapped_not_template - rast_rxns
mnt_ec_only = (mapped_not_template & mapped_ids) - transport_mapped
mnt_transport_only = (mapped_not_template & transport_mapped) - mapped_ids
mnt_both = mapped_not_template & mapped_ids & transport_mapped

nt_ec_df = ec_to_rxn[ec_to_rxn['rxn_bare'].isin(mapped_not_template)]
nt_ecs = set(nt_ec_df['ec'].unique())
nt_rast_ecs = nt_ecs & rast_ecs
nt_non_rast_ecs = nt_ecs - rast_ecs

uni_native = pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet')
uni_ecs = set(uni_native['ec'].unique())
del uni_native

pang = pd.read_parquet(f'{DATA_DIR}/pangenome_gc_ec.parquet')
pang_ecs = set(pang['ec'].unique())
del pang

print('Non-Template Reactions With UniProt Evidence')
print('=' * 70)
print(f'Total mapped NOT in templates: {len(mapped_not_template):,}')
print(f'\nEvidence source breakdown:')
print(f'  EC-based only:    {len(mnt_ec_only):>6,}')
print(f'  Transport only:   {len(mnt_transport_only):>6,}')
print(f'  Both:             {len(mnt_both):>6,}')
print(f'\nDetection feasibility (for adding to models):')
print(f'  RAST-detectable (same ECs exist in RAST): {len(mnt_rast):,} ({100*len(mnt_rast)/len(mapped_not_template):.1f}%)')
print(f'  Non-RAST (need alternative detection):    {len(mnt_no_rast):,} ({100*len(mnt_no_rast)/len(mapped_not_template):.1f}%)')
print(f'\nEC coverage for non-template mapped reactions:')
print(f'  Unique ECs: {len(nt_ecs):,}')
print(f'    In RAST:          {len(nt_rast_ecs):,}')
print(f'    Not in RAST:      {len(nt_non_rast_ecs):,}')
print(f'    In UniProt Tier1: {len(nt_ecs & uni_ecs):,}')
print(f'    In pangenome:     {len(nt_ecs & pang_ecs):,}')

Non-Template Reactions With UniProt Evidence
Total mapped NOT in templates: 15,338

Evidence source breakdown:
  EC-based only:    11,367
  Transport only:    3,317
  Both:                654

Detection feasibility (for adding to models):
  RAST-detectable (same ECs exist in RAST): 7,240 (47.2%)
  Non-RAST (need alternative detection):    8,098 (52.8%)

EC coverage for non-template mapped reactions:
  Unique ECs: 4,281
    In RAST:          1,704
    Not in RAST:      2,577
    In UniProt Tier1: 4,035
    In pangenome:     3,073


In [7]:
print('Detection Strategies for Non-Template Reactions')
print('=' * 70)
print()
print('For reactions where UniProt maps a protein but the reaction is NOT in')
print('any reconstruction template, how can we detect the enzyme in a genome?')
print()
print(f'1. RAST already covers it ({len(mnt_rast):,} reactions, {100*len(mnt_rast)/len(mapped_not_template):.1f}%)')
print(f'   The EC exists in RAST — enzyme would be found during annotation.')
print(f'   Bottleneck: template does not include the reaction, not detection.')
print()
print(f'2. HMM/BLAST vs UniProt reference proteins ({len(nt_non_rast_ecs):,} ECs not in RAST)')
print(f'   Build profiles from UniProt proteins annotated with these ECs.')
print(f'   Scan genome protein sequences against profiles.')
print(f'   UniProt Tier 1 covers {len(nt_ecs & uni_ecs):,} of these ECs.')
print()
print(f'3. eggNOG orthologous groups ({len(nt_ecs & pang_ecs):,} ECs in pangenome)')
print(f'   Use eggNOG OGs that carry these ECs — pre-computed from pangenome.')
print(f'   Run eggNOG-mapper on genome to get OG assignments, then EC transfer.')
print()
print(f'4. Transport substrate matching ({len(mnt_transport_only):,} transport-only reactions)')
print(f'   Substrate-specific GO/domain annotation (the NB11 approach).')
print(f'   Detect via InterPro domain signatures + GO term annotation.')

Detection Strategies for Non-Template Reactions

For reactions where UniProt maps a protein but the reaction is NOT in
any reconstruction template, how can we detect the enzyme in a genome?

1. RAST already covers it (7,240 reactions, 47.2%)
   The EC exists in RAST — enzyme would be found during annotation.
   Bottleneck: template does not include the reaction, not detection.

2. HMM/BLAST vs UniProt reference proteins (2,577 ECs not in RAST)
   Build profiles from UniProt proteins annotated with these ECs.
   Scan genome protein sequences against profiles.
   UniProt Tier 1 covers 4,035 of these ECs.

3. eggNOG orthologous groups (3,073 ECs in pangenome)
   Use eggNOG OGs that carry these ECs — pre-computed from pangenome.
   Run eggNOG-mapper on genome to get OG assignments, then EC transfer.

4. Transport substrate matching (3,317 transport-only reactions)
   Substrate-specific GO/domain annotation (the NB11 approach).
   Detect via InterPro domain signatures + GO term annotation.


## 6. Per-Template Breakdown

Compare coverage across GramNeg, GramPos, Archaea, and Core templates.

In [8]:
print('Per-Template Coverage Summary')
print('=' * 90)
print(f'{"Template":<12} {"Total":>6} {"Balanced":>9} {"Evidence":>9} {"Ev%":>7} '
      f'{"RAST":>6} {"Gap":>6} {"Gap%":>6}')
print('-' * 90)

for label in ['Core', 'GramNeg', 'GramPos', 'Archaea']:
    rxns = set()
    for rtype_set in per_template_types[label].values():
        rxns |= rtype_set
    bal = rxns & balanced_ids
    ev = rxns & all_mapped
    rast = rxns & rast_rxns
    gap = bal - all_mapped
    ev_pct = 100 * len(ev) / len(rxns) if rxns else 0
    gap_pct = 100 * len(gap) / len(bal) if bal else 0
    print(f'{label:<12} {len(rxns):>6} {len(bal):>9} {len(ev):>9} {ev_pct:>6.1f}% '
          f'{len(rast):>6} {len(gap):>6} {gap_pct:>5.1f}%')

    for rtype in ['conditional', 'gapfilling', 'spontaneous', 'universal']:
        tr = per_template_types[label].get(rtype, set())
        if not tr:
            continue
        tr_ev = tr & all_mapped
        tr_pct = 100 * len(tr_ev) / len(tr)
        print(f'  {rtype:<10} {len(tr):>6} {len(tr & balanced_ids):>9} '
              f'{len(tr_ev):>9} {tr_pct:>6.1f}%')

Per-Template Coverage Summary
Template      Total  Balanced  Evidence     Ev%   RAST    Gap   Gap%
------------------------------------------------------------------------------------------
Core            252       234       214   84.9%    160     20   8.5%
  conditional    230       212       201   87.4%
  gapfilling      4         4         4  100.0%
  spontaneous     13        13         5   38.5%
  universal       5         5         4   80.0%
GramNeg        8584      7851      5906   68.8%   3483   1945  24.8%
  conditional   6284      5752      5033   80.1%
  gapfilling   2258      2059       858   38.0%
  spontaneous     31        29         8   25.8%
  universal      11        11         7   63.6%
GramPos        8584      7851      5906   68.8%   3483   1945  24.8%
  conditional   6284      5752      5033   80.1%
  gapfilling   2258      2059       858   38.0%
  spontaneous     31        29         8   25.8%
  universal      11        11         7   63.6%
Archaea        8600  

## 7. Complex & Role Analysis

Examine the reaction→complex→role chain in templates. Roles carry functional
descriptions (often with EC numbers) that define what RAST looks for.

In [9]:
import re

with open(f'{TEMPLATE_DIR}/GramNegModelTemplateV7.json') as f:
    gram_neg = json.load(f)

role_lookup = {role['id']: role['name'] for role in gram_neg['roles']}
ec_pattern = re.compile(r'EC\s+(\d+\.\d+\.\d+\.[-\d]+)')
role_ecs = {}
for role in gram_neg['roles']:
    ecs = ec_pattern.findall(role.get('name', ''))
    if ecs:
        role_ecs[role['id']] = ecs

complex_roles = {}
for c in gram_neg['complexes']:
    rids = [cr['templaterole_ref'].split('/')[-1]
            for cr in c.get('complexroles', [])]
    complex_roles[c['id']] = rids

rxn_complex_refs = {}
for r in gram_neg['reactions']:
    bare = r['id'].rsplit('_', 1)[0]
    cids = [ref.split('/')[-1] for ref in r.get('templatecomplex_refs', [])]
    rxn_complex_refs[bare] = cids

n_with_complex = sum(1 for cids in rxn_complex_refs.values() if cids)
n_without_complex = sum(1 for cids in rxn_complex_refs.values() if not cids)

n_with_ec_role = 0
n_no_ec_role = 0
n_no_complex = 0
for bare, cids in rxn_complex_refs.items():
    if not cids:
        n_no_complex += 1
        continue
    has_ec = any(rid in role_ecs for cid in cids for rid in complex_roles.get(cid, []))
    if has_ec:
        n_with_ec_role += 1
    else:
        n_no_ec_role += 1

print('Reaction → Complex → Role → EC Chain (GramNeg template)')
print('=' * 60)
print(f'Roles total: {len(gram_neg["roles"]):,}')
print(f'Roles with EC in name: {len(role_ecs):,} ({100*len(role_ecs)/len(gram_neg["roles"]):.1f}%)')
print(f'Roles without EC: {len(gram_neg["roles"]) - len(role_ecs):,}')
print(f'\nReactions with complex refs: {n_with_complex:,}')
print(f'Reactions without complex refs: {n_without_complex:,}')
print(f'\nReactions with EC-bearing role: {n_with_ec_role:,}')
print(f'Reactions with complex but no EC in roles: {n_no_ec_role:,}')
print(f'Reactions with no complex: {n_no_complex:,}')

print(f'\nBy type — reactions WITH complex refs:')
for rtype in ['conditional', 'gapfilling', 'spontaneous', 'universal']:
    with_c = sum(1 for r in gram_neg['reactions']
                 if r['type'] == rtype and r.get('templatecomplex_refs'))
    without_c = sum(1 for r in gram_neg['reactions']
                    if r['type'] == rtype and not r.get('templatecomplex_refs'))
    print(f'  {rtype}: {with_c} with, {without_c} without')

Reaction → Complex → Role → EC Chain (GramNeg template)
Roles total: 20,548
Roles with EC in name: 8,047 (39.2%)
Roles without EC: 12,501

Reactions with complex refs: 2,719
Reactions without complex refs: 5,865

Reactions with EC-bearing role: 2,295
Reactions with complex but no EC in roles: 424
Reactions with no complex: 5,865

By type — reactions WITH complex refs:
  conditional: 2590 with, 3694 without
  gapfilling: 119 with, 2139 without
  spontaneous: 5 with, 26 without
  universal: 5 with, 6 without


## 8. Figures

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Coverage by reaction type
ax = axes[0]
types = ['conditional', 'gapfilling', 'spontaneous', 'universal']
x = np.arange(len(types))
width = 0.35

totals_by_type = [len(type_sets[t] & balanced_ids) for t in types]
evidence_by_type = [len(type_sets[t] & all_mapped) for t in types]

bars1 = ax.bar(x - width/2, totals_by_type, width, label='Balanced', color='#bab0ac')
bars2 = ax.bar(x + width/2, evidence_by_type, width, label='With evidence', color='#4c78a8')

ax.set_xticks(x)
ax.set_xticklabels([t.capitalize() for t in types], rotation=30, ha='right')
ax.set_ylabel('Reactions')
ax.set_title('A. Template Coverage by Type')
ax.legend(loc='upper right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

for b1, b2 in zip(bars1, bars2):
    h1, h2 = b1.get_height(), b2.get_height()
    if h1 > 0:
        pct = 100 * h2 / h1
        ax.text(b2.get_x() + b2.get_width()/2, h2 + 30,
                f'{pct:.0f}%', ha='center', va='bottom', fontsize=8)

# Panel B: Gap breakdown for conditional vs gapfilling
ax = axes[1]
gap_cond = (type_sets['conditional'] & balanced_ids) - all_mapped
gap_gapf = (type_sets['gapfilling'] & balanced_ids) - all_mapped

categories = ['Has EC,\nno protein', 'No EC\n(orphan)', 'Transport', 'Non-transport']
cond_vals = [
    len(gap_cond & all_ec_rxns),
    len(gap_cond - all_ec_rxns),
    len(gap_cond & transport_rxn_ids),
    len(gap_cond - transport_rxn_ids),
]
gapf_vals = [
    len(gap_gapf & all_ec_rxns),
    len(gap_gapf - all_ec_rxns),
    len(gap_gapf & transport_rxn_ids),
    len(gap_gapf - transport_rxn_ids),
]

x2 = np.arange(len(categories))
ax.bar(x2 - width/2, cond_vals, width, label='Conditional', color='#4c78a8')
ax.bar(x2 + width/2, gapf_vals, width, label='Gapfilling', color='#f58518')
ax.set_xticks(x2)
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylabel('Gap Reactions')
ax.set_title('B. Gap Characterization')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# Panel C: Template vs non-template evidence
ax = axes[2]
in_template_ev = len(all_template_rxns & all_mapped)
in_template_gap = len((all_template_rxns & balanced_ids) - all_mapped)
not_template_ev = len(mapped_not_template)
remaining = len(balanced_ids) - in_template_ev - in_template_gap - not_template_ev

slices = [in_template_ev, not_template_ev, in_template_gap, remaining]
labels = [
    f'Template + evidence\n({in_template_ev:,})',
    f'Non-template + evidence\n({not_template_ev:,})',
    f'Template gap\n({in_template_gap:,})',
    f'No template, no evidence\n({remaining:,})',
]
colors = ['#4c78a8', '#72b7b2', '#f58518', '#bab0ac']
ax.pie(slices, labels=labels, colors=colors, autopct='%1.1f%%',
       pctdistance=0.75, labeldistance=1.2, textprops={'fontsize': 8})
ax.set_title('C. Balanced Reaction Landscape')

fig.suptitle('NB12: ModelSEED Template Reaction Coverage', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/template_coverage.png')
plt.show()
print('Saved: template_coverage.png')

Saved: template_coverage.png


## 9. Save Summary & Output

In [11]:
coverage_df.to_csv(f'{DATA_DIR}/template_coverage_summary.tsv', sep='\t', index=False)
print(f'Saved: template_coverage_summary.tsv ({len(coverage_df)} rows)')

print()
print('=' * 70)
print('NB12 TEMPLATE COVERAGE SUMMARY')
print('=' * 70)
print(f'\nTemplates parsed: {len(template_files)} (v7.0)')
print(f'  GramNeg/GramPos: identical reaction sets (8,584 each)')
print(f'  Archaea: 8,600 (16 unique beyond GramNeg/Pos)')
print(f'  Core: 252 (subset of GramNeg/Pos)')
print(f'  Union: {len(all_template_rxns):,} unique reactions')
print(f'\nBalanced ModelSEED reactions: {len(balanced_ids):,}')
print(f'Templates cover: {len(all_template_rxns & balanced_ids):,} / {len(balanced_ids):,} '
      f'({100*len(all_template_rxns & balanced_ids)/len(balanced_ids):.1f}%) of balanced reactions')
print(f'\n--- Coverage by Type ---')
for rtype in ['conditional', 'gapfilling', 'spontaneous', 'universal']:
    rxns = type_sets[rtype]
    ev = rxns & all_mapped
    bal = rxns & balanced_ids
    gap = bal - all_mapped
    print(f'  {rtype}: {len(ev):,} / {len(rxns):,} with evidence ({100*len(ev)/len(rxns):.1f}%), '
          f'{len(gap):,} gap')
print(f'\n--- Key Findings ---')
print(f'1. Conditional reactions: 80% covered — gene-associated, well-characterized')
print(f'2. Gapfilling reactions: 38% covered — computationally added, mostly EC orphans')
print(f'   ({len((type_sets["gapfilling"] & balanced_ids) - all_mapped - all_ec_rxns):,} / '
      f'{len((type_sets["gapfilling"] & balanced_ids) - all_mapped):,} gap reactions have no EC)')
print(f'3. {len(mapped_not_template):,} reactions have UniProt evidence but are NOT in templates')
print(f'   {len(mnt_rast):,} ({100*len(mnt_rast)/len(mapped_not_template):.1f}%) RAST-detectable — '
      f'template is the bottleneck, not detection')
print(f'   {len(mnt_no_rast):,} ({100*len(mnt_no_rast)/len(mapped_not_template):.1f}%) need '
      f'HMM/BLAST or domain-based detection')

Saved: template_coverage_summary.tsv (5 rows)

NB12 TEMPLATE COVERAGE SUMMARY

Templates parsed: 4 (v7.0)
  GramNeg/GramPos: identical reaction sets (8,584 each)
  Archaea: 8,600 (16 unique beyond GramNeg/Pos)
  Core: 252 (subset of GramNeg/Pos)
  Union: 8,606 unique reactions

Balanced ModelSEED reactions: 34,343
Templates cover: 7,869 / 34,343 (22.9%) of balanced reactions

--- Coverage by Type ---
  conditional: 5,043 / 6,306 with evidence (80.0%), 727 gap
  gapfilling: 858 / 2,258 with evidence (38.0%), 1,201 gap
  spontaneous: 8 / 31 with evidence (25.8%), 21 gap
  universal: 7 / 11 with evidence (63.6%), 4 gap

--- Key Findings ---
1. Conditional reactions: 80% covered — gene-associated, well-characterized
2. Gapfilling reactions: 38% covered — computationally added, mostly EC orphans
   (1,182 / 1,201 gap reactions have no EC)
3. 15,338 reactions have UniProt evidence but are NOT in templates
   7,240 (47.2%) RAST-detectable — template is the bottleneck, not detection
   8,098 (